# Задание 7.2: Уравнение Бюргерса

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib.animation as anim
from IPython.display import HTML
from scipy.linalg import solve_banded

# Параметры

In [ ]:
L = 1 # длина струны
nu = 0.0 # параметр уравнения

t0 = 0 # нач момент
t1 = 1 # конечный момент

dt = 0.0001 # шаг по времени

# Параметры моделирования

In [ ]:
Nx = 201 # кол-во вершин

dx = L/(Nx-1) # шаг по пространству

x = np.linspace(0,L,Nx)

u0 = np.sin( 2*np.pi*x )

# Параметры анимации

In [ ]:
anim_time = 5 # время на анимацию
fps = 20 # кол-во кадров в ссекунду

## Анимация

In [ ]:
def anime( u , fps=fps , anim_time=anim_time ):
    step = 1 + math.floor(len(u)/(fps*anim_time))
    u = u[::step,]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    line, = ax.plot(x, u[0], 'b-', linewidth=2)
    ax.set_xlim(0, L)
    # ax.set_ylim(u.min(), u.max())
    ax.set_xlabel('Позиция x')
    ax.set_ylabel('Значение u')
    ax.set_title(f'')
    ax.grid(True)
    
    def update(frame):
        line.set_ydata(u[frame, :])
        return line,
    
    animt = anim.FuncAnimation(
        fig, 
        update, 
        frames=len(u),
        interval=1000/fps,  
        blit=True,
        repeat=True
    )
    
    plt.close()
    return HTML(animt.to_jshtml())

## Схема Upwind

In [ ]:
def upwind( u0 , nu , dt , dx , t0 , t1 ):
    Nx = len(u0)
    Nt = math.floor((t1-t0)/dt) #кол-во шагов равных dt
    dt_last = t1 - t0 - Nt*dt # последний шаг, который может быть меньше, так как надо точно попасть в t1

    t = np.linspace(t0,t1,Nt+1)
    if math.fabs(dt_last) > 1.e-8:
        t.append(t[-1]+dt_last)

    if math.fabs(dt_last) > 1.e-8:
        u = np.zeros( ( 1+Nt+1 , Nx ) )
    else:
        u = np.zeros( ( 1+Nt , Nx ) )

    u[0] = u0

    for j in range(0,Nt):
        ul = np.roll( u[j],1 )
        ur = np.roll( u[j],-1 )
        F = np.where( u[j] >= 0, u[j] * (u[j] - ul), u[j] * (ur - u[j]))
        u[j+1] = u[j] - dt*( F/dx - nu*( ur - 2*u[j] + ul )/dx**2 )

    if math.fabs(dt_last) > 1.e-8:
        ul = np.roll( u[-2],1 )
        ur = np.roll( u[-2],-1 )
        F = np.where( u[-2] >= 0, u[-2] * (u[-2] - ul), u[-2] * (ur - u[-2]))
        u[-1] = u[-2] - dt*( F/dx - nu*( ur - 2*u[-2] + ul )/dx**2 )
    return (u , t)

## Схема Лакса-Вендрофа

In [ ]:
def lax_wend( u0 , nu , dt , dx , t0 , t1 ):
    Nx = len(u0)
    Nt = math.floor((t1-t0)/dt) #кол-во шагов равных dt
    dt_last = t1 - t0 - Nt*dt # последний шаг, который может быть меньше, так как надо точно попасть в t1

    t = np.linspace(t0,t1,Nt+1)
    if math.fabs(dt_last) > 1.e-8:
        t.append(t[-1]+dt_last)

    if math.fabs(dt_last) > 1.e-8:
        u = np.zeros( ( 1+Nt+1 , Nx ) )
    else:
        u = np.zeros( ( 1+Nt , Nx ) )

    u[0] = u0

    for j in range(Nt):
        u_cur = u[j]
        
        u_left = np.roll(u_cur, 1)
        u_right = np.roll(u_cur, -1)

        u[j+1] = u_cur - 0.5 * dt/dx * u_cur * ( u_right - u_left ) + (0.5 * dt/dx * u_cur)**2 * (u_right - 2* u_cur + u_left)
        
        u[j+1] += nu * dt / dx**2 * (u_right - 2*u_cur + u_left)

    if math.fabs(dt_last) > 1.e-8:
        j = -2
        u_cur = u[j]
        
        u_left = np.roll(u_cur, 1)
        u_right = np.roll(u_cur, -1)
        
        u[j+1] = u_cur - 0.5 * dt/dx * u_cur * ( u_right - u_left ) + (0.5 * dt/dx * u_cur)**2 * (u_right - 2* u_cur + u_left)
        u[j+1] += nu * dt / dx**2 * (u_right - 2*u_cur + u_left)

    return (u , t)

# Зелёный уровень

## Вычисление Лакс-Вендроф

In [ ]:
u_lw , t_lw = lax_wend( u0 , nu , dt , dx , t0 , t1 )
anime( u_lw , fps , anim_time )

## Вычисление Up-Wind

In [ ]:
u_upwind , l_wind = upwind( u0 , nu , dt , dx , t0 , t1 )
anime( u_upwind , fps , anim_time )

## Сравнение методов

In [ ]:
import matplotlib.pyplot as plt

# Сравнение в момент разрыва
def compare_methods(u_upwind, u_lw, x, t):
    # Найти момент с максимальным градиентом
    grad_max = 0
    t_idx = 0
    
    for i in range(len(u_upwind)):
        grad = np.max(np.abs(np.gradient(u_upwind[i], x)))
        if grad > grad_max:
            grad_max = grad
            t_idx = i
    
    print(f"Максимальный градиент в t = {t[t_idx]:.3f}")
    
    # Графики
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(x, u_upwind[t_idx], 'b-', label='Upwind', linewidth=2)
    plt.plot(x, u_lw[t_idx], 'r--', label='Lax-Wendroff', linewidth=2)
    plt.title('Сравнение диссипации')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    # Увеличение вблизи разрыва
    shock_pos = np.argmax(np.abs(np.gradient(u_upwind[t_idx], x)))
    idx_start = max(0, shock_pos - 15)
    idx_end = min(len(x), shock_pos + 15)
    
    plt.plot(x[idx_start:idx_end], u_upwind[t_idx][idx_start:idx_end], 'b-', label='Upwind', linewidth=2)
    plt.plot(x[idx_start:idx_end], u_lw[t_idx][idx_start:idx_end], 'r--', label='Lax-Wendroff', linewidth=2)
    plt.title('Сравнение диссипации')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Анализ
    region = u_lw[t_idx][idx_start:idx_end]
    oscillations = np.std(region - np.mean(region))
    print(f"Осцилляции Lax-Wendroff: {oscillations:.6f}")

# Простое использование
compare_methods(u_upwind, u_lw, x, t_lw)

# Жёлтый уровень

In [ ]:
nu = 0.005

## Вычисление Лакс-Вендроф

In [ ]:
u_lw , t_lw = lax_wend( u0 , nu , dt , dx , t0 , t1 )
anime( u_lw , fps , anim_time )

## Вычисление Up-Wind

In [ ]:
u_upwind , l_wind = upwind( u0 , nu , dt , dx , t0 , t1 )
anime( u_upwind , fps , anim_time )

## Вычисление ширины фронта

In [ ]:
def w_front(u,dx):
    i_min = np.argmin(u)
    i_max = np.argmax(u)
    return np.abs(dx*(i_max-i_min))

## График зависимости ширины фронта от вязкости

In [ ]:
nuu = [0.002*(i+1) for i in range(10)]

for nu in nuu:
    w = []
    u_upwind , t_wind = upwind( u0 , nu , dt , dx , t0 , t1 )
    print(len(u_upwind) , len(t_wind) , nu)
    for i in range( 0 , len(u_upwind) , 100 ):
        w.append( w_front(u_upwind[i],dx) )
    plt.plot( t_wind[::100] , w , "-" , label = f"w , nu = {nu}" )

plt.xlabel('T')
plt.ylabel(f'$\delta$')
plt.title(f'Изменения ширины фронта со временем')
plt.grid(True)
plt.legend()
plt.show()

## Сравнение с известным решением

In [ ]:
def anime2( u , u_an , fps=fps , anim_time=anim_time ):
    step = 1 + math.floor(len(u)/(fps*anim_time))
    u = u[::step,]
    u_an = u_an[::step,]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    line, = ax.plot(x, u[0], 'b-', linewidth=2,label=f"Analit")
    line_an, = ax.plot(x, u_an[0], 'r-', linewidth=2,label=f"up-wind")
    ax.set_xlim(0, L)
    # ax.set_ylim(u.min(), u.max())
    ax.set_xlabel('Позиция x')
    ax.set_ylabel('Значение u')
    ax.set_title(f'')
    ax.grid(True)
    ax.legend()
    
    def update(frame):
        line.set_ydata(u[frame, :])
        line_an.set_ydata(u_an[frame, :])
        return (line , line_an)
    
    animt = anim.FuncAnimation(
        fig, 
        update, 
        frames=len(u),
        interval=1000/fps,  
        blit=True,
        repeat=True
    )
    
    plt.close()
    return HTML(animt.to_jshtml())

In [ ]:
U = 1.0
s = 0.3
delta = 0.08

nu = U*delta/2

u0 = -U * np.tanh( x / delta)


u_upwind , t_wind = upwind( u0 , nu , dt , dx , t0 , t1 )

u_an = np.zeros( ( len(t_wind) , Nx ) )
for j in range(len(t_wind)):
    for i in range(Nx):
        u_an[j,i] = -U * np.tanh( (x[i]-s*t_wind[j]) / delta)

anime2( u_upwind , u_an )